[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhangshuo-zh/AI_Video_Platform/blob/main/tools/colab_comfyui_flux_kontext.ipynb)

# AI 视频生产平台 · Colab ComfyUI L4 22.5G 专享版（FLUX.1-Kontext-dev FP8）

专为 **L4 22.5 GB 显存** 与 **角色跨视角一致性（先出正面定妆，再转侧面/背面/特写）** 打造的一键启动笔记本：
1. **吃满 L4 算力**：运行 12B 参数级 `FLUX.1-Kontext-dev` FP8 + `T5-XXL FP8` + `CLIP-L` + `Flux VAE`，构图遵循度、手部与细节远超 SDXL。
2. **先开箱即用**：点下方 **「1. 一键启动」**，自动拉取 ComfyUI（含 `ReferenceLatent` 节点）并下载 FLUX Kontext 全部权重（首次约需 3～5 分钟，全部免登录直连下载）。
3. **默认免 Drive 授权弹窗**：`USE_GOOGLE_DRIVE` 默认关闭，全程无人值守免 OAuth 弹窗。
4. **共用域名注意**：与动漫/2.5D 版共用 `NGROK_DOMAIN`，同一时间只开 1 个笔记本即可。
5. **闲置自动停表**：连续 **15 分钟**无新任务自动释放运行时（`runtime.unassign()`）。

In [ ]:
#@title 1. 一键启动 ComfyUI FLUX.1-Kontext 版（L4 22.5G 专享 + 免弹窗 + 闲置自动关机）
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_FLUX_KONTEXT = True #@param {type:"boolean"}
DOWNLOAD_ANIME_UPSCALE = True #@param {type:"boolean"}
DOWNLOAD_REALISTIC_UPSCALE = True #@param {type:"boolean"}
IDLE_SHUTDOWN_MINUTES = 15 #@param {type:"integer"}
NGROK_DOMAIN = "pushchair-decimeter-bath.ngrok-free.dev" #@param {type:"string"}
NGROK_AUTHTOKEN = "" #@param {type:"string"}
EXTRA_MODEL_URL = "" #@param {type:"string"}
EXTRA_MODEL_SUBDIR = "checkpoints" #@param ["checkpoints", "loras", "upscale_models", "vae", "controlnet", "diffusion_models", "text_encoders"]
EXTRA_MODEL_FILENAME = "" #@param {type:"string"}

import os
import re
import json
import time
import shutil
import threading
import subprocess
import urllib.request
from pathlib import Path

COMFY_DIR = Path('/content/ComfyUI')
LOG_PATH = Path('/tmp/comfyui.log')

# --- 第 1 步：准备 ComfyUI 与模型目录（按开关决定是否挂载 Google Drive） ---
if not COMFY_DIR.exists():
    print('📦 首次初始化 ComfyUI（约 1–2 分钟）...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', str(COMFY_DIR / 'requirements.txt')], check=True)

models_link = COMFY_DIR / 'models'
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    DRIVE_ROOT = Path('/content/drive/MyDrive/AI_Video_Platform_ComfyUI')
    MODELS_DIR = DRIVE_ROOT / 'models'
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    (DRIVE_ROOT / 'workflows').mkdir(parents=True, exist_ok=True)
    if models_link.exists() and not models_link.is_symlink():
        shutil.rmtree(models_link)
    if not models_link.exists():
        models_link.symlink_to(MODELS_DIR)
else:
    print('⚡ 免授权模式：跳过 Google Drive 挂载，模型将直接下载到 Colab 本地盘')
    if models_link.is_symlink():
        models_link.unlink()
    MODELS_DIR = models_link

for sub in ['checkpoints', 'loras', 'upscale_models', 'vae', 'controlnet', 'clip', 'clip_vision', 'unet', 'diffusion_models', 'text_encoders']:
    (MODELS_DIR / sub).mkdir(parents=True, exist_ok=True)

def read_secret(key: str) -> str:
    try:
        from google.colab import userdata
        return (userdata.get(key) or '').strip()
    except Exception:
        return ''

ngrok_token = NGROK_AUTHTOKEN.strip() or read_secret('NGROK_AUTHTOKEN')
ngrok_domain = NGROK_DOMAIN.strip() or read_secret('NGROK_DOMAIN')
use_ngrok = bool(ngrok_token and ngrok_domain)

if use_ngrok:
    if not shutil.which('ngrok'):
        subprocess.run('curl -sSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz | tar -xz -C /usr/local/bin', shell=True, check=True)
    subprocess.run(['ngrok', 'config', 'add-authtoken', ngrok_token], check=True)
elif not shutil.which('cloudflared'):
    subprocess.run(['curl', '-sL', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb', '-o', '/tmp/cloudflared.deb'], check=True)
    subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True)

# --- 第 2 步：按需下载模型到 Google Drive（已存在自动跳过） ---
def ensure_file(url: str, dest: Path, min_bytes: int = 1024 * 1024):
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print(f'⏭️  模型已就绪，跳过下载：{dest.name} ({dest.stat().st_size / 1024 / 1024:.1f} MB)')
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f'⬇️  正在下载到 Google Drive：{dest.name}（首次约需 1–3 分钟，后续永久秒加载）...')
    subprocess.run(['curl', '-fL', '-C', '-', url, '-o', str(dest)], check=True)
    print(f'✅ 下载完成：{dest.name} ({dest.stat().st_size / 1024 / 1024:.1f} MB)')

if DOWNLOAD_ANIME_UPSCALE:
    ensure_file(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
        MODELS_DIR / 'upscale_models' / 'RealESRGAN_x4plus_anime_6B.pth',
        min_bytes=10 * 1024 * 1024,
    )

if DOWNLOAD_REALISTIC_UPSCALE:
    ensure_file(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        MODELS_DIR / 'upscale_models' / 'RealESRGAN_x4plus.pth',
        min_bytes=10 * 1024 * 1024,
    )

if DOWNLOAD_FLUX_KONTEXT:
    ensure_file(
        'https://huggingface.co/Comfy-Org/Flux-1-Kontext-Dev/resolve/main/split_files/diffusion_models/flux1-dev-kontext_fp8_scaled.safetensors',
        MODELS_DIR / 'diffusion_models' / 'flux1-dev-kontext_fp8_scaled.safetensors',
        min_bytes=1024 * 1024 * 1024,
    )
    ensure_file(
        'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        MODELS_DIR / 'text_encoders' / 'clip_l.safetensors',
        min_bytes=10 * 1024 * 1024,
    )
    ensure_file(
        'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn_scaled.safetensors',
        MODELS_DIR / 'text_encoders' / 't5xxl_fp8_e4m3fn_scaled.safetensors',
        min_bytes=1024 * 1024 * 1024,
    )
    ensure_file(
        'https://huggingface.co/Comfy-Org/Lumina_Image_2.0_Repackaged/resolve/main/split_files/vae/ae.safetensors',
        MODELS_DIR / 'vae' / 'ae.safetensors',
        min_bytes=10 * 1024 * 1024,
    )

if EXTRA_MODEL_URL.strip():
    fname = EXTRA_MODEL_FILENAME.strip() or EXTRA_MODEL_URL.strip().split('?')[0].rstrip('/').split('/')[-1]
    ensure_file(EXTRA_MODEL_URL.strip(), MODELS_DIR / EXTRA_MODEL_SUBDIR / fname, min_bytes=1024)

# --- 第 3 步：启动 ComfyUI + Cloudflare 隧道 + 闲置自动关机守护 ---
subprocess.run(['pkill', '-f', 'ComfyUI/main.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
subprocess.run(['pkill', '-f', 'ngrok'], check=False)
time.sleep(1)

log_file = open(LOG_PATH, 'w')
comfy_proc = subprocess.Popen(
    ['python3', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--dont-print-server'],
    cwd=str(COMFY_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print('🚀 正在启动 ComfyUI，等待 8188 端口就绪...')
ready = False
for _ in range(180):
    if comfy_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    log_file.flush()
    tail = LOG_PATH.read_text(errors='ignore')[-3000:] if LOG_PATH.exists() else ''
    raise RuntimeError(f'ComfyUI 启动失败，日志末尾如下：\n{tail}')
print('✅ ComfyUI 已就绪！正在建立 HTTPS 隧道...')

def idle_watchdog(minutes: int):
    if minutes <= 0:
        return
    last_active = time.time()
    last_history_len = 0
    while True:
        time.sleep(20)
        try:
            with urllib.request.urlopen('http://127.0.0.1:8188/queue', timeout=3) as r:
                q = json.loads(r.read())
            if q.get('queue_running') or q.get('queue_pending'):
                last_active = time.time()
                continue
            with urllib.request.urlopen('http://127.0.0.1:8188/history', timeout=3) as r:
                hist = json.loads(r.read())
            if len(hist) != last_history_len:
                last_history_len = len(hist)
                last_active = time.time()
        except Exception:
            pass
        idle_sec = time.time() - last_active
        if idle_sec >= minutes * 60:
            print(f'\n🛑 已连续闲置 {minutes} 分钟，自动断开并释放 Colab GPU 算力单元！')
            try:
                from google.colab import runtime
                runtime.unassign()
            except Exception:
                os._exit(0)
            return

threading.Thread(target=idle_watchdog, args=(IDLE_SHUTDOWN_MINUTES,), daemon=True).start()

if use_ngrok:
    public_url = f'https://{ngrok_domain}'
    print('\n' + '=' * 64)
    print('🌐 ComfyUI 固定永久地址（已在平台配置一次即可长期复用）：')
    print(f'   {public_url}')
    print(f'⏱️  省算力守护已开启：连续闲置 {IDLE_SHUTDOWN_MINUTES} 分钟后自动释放运行时')
    print('=' * 64 + '\n')
    tunnel = subprocess.Popen(
        ['ngrok', 'http', '127.0.0.1:8188', '--domain', ngrok_domain, '--log', 'stdout'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in tunnel.stdout:
        if 'err=' in line or 'ERROR' in line:
            print(line, end='')
else:
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    url_pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    public_url = None
    for line in tunnel.stdout:
        m = url_pattern.search(line)
        if m and not public_url:
            public_url = m.group(0)
            print('\n' + '=' * 64)
            print('🌐 ComfyUI 临时公网地址（填进平台「模型中心 → 引擎实例 → ComfyUI」）：')
            print(f'   {public_url}')
            print(f'⏱️  省算力守护已开启：连续闲置 {IDLE_SHUTDOWN_MINUTES} 分钟后自动释放运行时')
            print('=' * 64 + '\n')


In [ ]:
#@title 2. 用完立刻关机（手动释放 Colab GPU，立刻停止扣除计算单元）
from google.colab import runtime
print('🛑 正在主动释放 Colab GPU 运行时...')
runtime.unassign()
